# Text Feature Engineering Assignment
## Text Processing Pipeline — Real-world Dataset (Flipkart Product Reviews)

**Product:** Motorola Edge 60 Fusion 5G  
**Source:** Flipkart Product Reviews (scraped using Selenium + BeautifulSoup)  
**Total Reviews Collected:** 320 (270 after cleaning)

---
## Data Collection
Scraping product reviews from Flipkart using Selenium and BeautifulSoup.

In [1]:
# ---------- Imports for Web Scraping ----------
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import csv, time

In [2]:
# ---------- Scraping Configuration ----------
BASE_URL = (
    "https://www.flipkart.com/motorola-edge-60-fusion-5g-pantone-amazonite-128-gb/"
    "product-reviews/itm0ba5c1f57331a?pid=MOBHHD2KXMH9NCYG"
    "&lid=LSTMOBHHD2KXMH9NCYGOY7CC7&marketplace=FLIPKART&page="
)
PAGES = 50

In [3]:
# ---------- Browser Setup ----------
options = Options()
options.add_argument("--headless")
options.add_argument("--disable-gpu")
options.add_argument("--no-sandbox")
options.add_argument("user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64)")

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)

In [4]:
# ---------- Scrape Reviews ----------
reviews = []
for page in range(1, PAGES + 1):
    print(f"Scraping page {page}...")
    driver.get(BASE_URL + str(page))
    try:
        WebDriverWait(driver, 10).until(
            EC.presence_of_element_located((By.CLASS_NAME, "css-1jxf684"))
        )
    except:
        print(f"  Page {page} didn't load reviews, skipping.")
        continue
    soup = BeautifulSoup(driver.page_source, "html.parser")
    bodies = soup.find_all("span", class_="css-1jxf684")
    for body in bodies:
        reviews.append({"review_text": body.text.strip()})
    print(f"  Found {len(bodies)} reviews")
    time.sleep(2)

driver.quit()
print(f"\nDone! Collected {len(reviews)} reviews.")

Scraping page 1...
  Found 13 reviews
Scraping page 2...
  Found 12 reviews
Scraping page 3...
  Found 10 reviews
Scraping page 4...
  Found 11 reviews
Scraping page 5...
  Found 12 reviews
Scraping page 6...
  Found 14 reviews
Scraping page 7...
  Found 10 reviews
Scraping page 8...
  Found 13 reviews
Scraping page 9...
  Found 13 reviews
Scraping page 10...
  Found 13 reviews
Scraping page 11...
  Found 4 reviews
Scraping page 12...
  Found 2 reviews
Scraping page 13...
  Found 8 reviews
Scraping page 14...
  Found 3 reviews
Scraping page 15...
  Found 5 reviews
Scraping page 16...
  Found 3 reviews
Scraping page 17...
  Found 6 reviews
Scraping page 18...
  Found 4 reviews
Scraping page 19...
  Found 6 reviews
Scraping page 20...
  Found 6 reviews
Scraping page 21...
  Found 4 reviews
Scraping page 22...
  Found 2 reviews
Scraping page 23...
  Found 5 reviews
Scraping page 24...
  Found 3 reviews
Scraping page 25...
  Found 1 reviews
Scraping page 26...
  Found 1 reviews
Scraping pa

In [5]:
# ---------- Save to CSV ----------
with open("flipkart_reviews.csv", "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["review_text"])
    writer.writeheader()
    writer.writerows(reviews)
print("Reviews saved to flipkart_reviews.csv")

Reviews saved to flipkart_reviews.csv


---
## Load and Clean the Dataset

In [7]:
import pandas as pd
import numpy as np
import string
import re

df = pd.read_csv("flipkart_reviews.csv")
print(f"Total rows loaded: {len(df)}")

# Basic cleaning
df['review_text'] = df['review_text'].str.replace('\n', ' ', regex=False)
df['review_text'] = df['review_text'].str.replace(r'[^a-zA-Z0-9\s]', '', regex=True)
df = df[df['review_text'].str.strip().str.len() > 5]
df = df[df['review_text'].str.lower() != 'more']
df = df.drop_duplicates(subset='review_text').reset_index(drop=True)

print(f"Clean reviews: {len(df)}")
df.head(10)

Total rows loaded: 275
Clean reviews: 223


,review_text
0,Good phone in this price range
1,Mobile looks extremely good and premium Camera...
2,Nice product
3,Nice phone
4,Good camera overall good phone
5,Everything is good but Motorola should also la...
6,Phone is good but there is less one gallery ap...
7,Very good image quality
8,One of the best phones in this price range Pro...
9,Sony lytia 700 Camera is superb Processor is...


---
## Task 1: Text Preprocessing
1. Convert to lowercase
2. Tokenization
3. Remove punctuation
4. Remove stopwords
5. Lemmatization (optional)

In [8]:
# Step 1 & 2: Lowercase + Tokenization
tokens = [sentence.lower().split() for sentence in df["review_text"].tolist()]
print("Sample tokens (first 3 reviews):")
for i, t in enumerate(tokens[:3]):
    print(f"  Review {i}: {t}")

Sample tokens (first 3 reviews):
  Review 0: ['good', 'phone', 'in', 'this', 'price', 'range']
  Review 1: ['mobile', 'looks', 'extremely', 'good', 'and', 'premium', 'camera', 'quality', 'is', 'exceptional', 'charges', 'quickly', 'drawback', 'battery', 'drains', 'quickly', 'despite', 'no', 'overuse', 'i', 'basically', 'use', 'the', 'phone', 'just', 'fr', 'calls', 'whats', 'app', 'n', 'so']
  Review 2: ['nice', 'product']


In [9]:
# Step 3: Remove Punctuation
clean_tokens = []
for sentence in tokens:
    clean = [word for word in sentence if word not in string.punctuation]
    clean_tokens.append(clean)

print("After punctuation removal (first 3):")
for i, t in enumerate(clean_tokens[:3]):
    print(f"  Review {i}: {t}")

After punctuation removal (first 3):
  Review 0: ['good', 'phone', 'in', 'this', 'price', 'range']
  Review 1: ['mobile', 'looks', 'extremely', 'good', 'and', 'premium', 'camera', 'quality', 'is', 'exceptional', 'charges', 'quickly', 'drawback', 'battery', 'drains', 'quickly', 'despite', 'no', 'overuse', 'i', 'basically', 'use', 'the', 'phone', 'just', 'fr', 'calls', 'whats', 'app', 'n', 'so']
  Review 2: ['nice', 'product']


In [10]:
# Step 4: Remove Stopwords
STOP_WORDS = {
    'i','me','my','myself','we','our','ours','ourselves','you','your','yours',
    'yourself','yourselves','he','him','his','himself','she','her','hers',
    'herself','it','its','itself','they','them','their','theirs','themselves',
    'what','which','who','whom','this','that','these','those','am','is','are',
    'was','were','be','been','being','have','has','had','having','do','does',
    'did','doing','a','an','the','and','but','if','or','because','as','until',
    'while','of','at','by','for','with','about','against','between','through',
    'during','before','after','above','below','to','from','up','down','in',
    'out','on','off','over','under','again','further','then','once','here',
    'there','when','where','why','how','all','both','each','few','more','most',
    'other','some','such','no','nor','not','only','own','same','so','than',
    'too','very','s','t','can','will','just','don','should','now','d','ll',
    'm','o','re','ve','y'
}

final_tokens = []
for sentence in clean_tokens:
    filtered = [word for word in sentence if word not in STOP_WORDS]
    final_tokens.append(filtered)

print("After stopword removal (first 5):")
for i, t in enumerate(final_tokens[:5]):
    print(f"  Review {i}: {t}")

After stopword removal (first 5):
  Review 0: ['good', 'phone', 'price', 'range']
  Review 1: ['mobile', 'looks', 'extremely', 'good', 'premium', 'camera', 'quality', 'exceptional', 'charges', 'quickly', 'drawback', 'battery', 'drains', 'quickly', 'despite', 'overuse', 'basically', 'use', 'phone', 'fr', 'calls', 'whats', 'app', 'n']
  Review 2: ['nice', 'product']
  Review 3: ['nice', 'phone']
  Review 4: ['good', 'camera', 'overall', 'good', 'phone']


---
## Task 2: Vocabulary Creation

In [11]:
from collections import Counter

all_words = [word for sentence in final_tokens for word in sentence]
vocab = sorted(set(all_words))
print(f"Vocabulary Size: {len(vocab)}")
print(f"Total word occurrences: {len(all_words)}")
print(f"\nFirst 20 words in vocabulary (alphabetical):")
print(vocab[:20])

Vocabulary Size: 666
Total word occurrences: 2011

First 20 words in vocabulary (alphabetical):
['0', '1', '10', '1010', '12', '13', '18k', '19k', '1camera', '1excellent', '1st', '2', '20000', '2017', '20k', '24', '25', '25k', '2premium', '3awesome']


In [12]:
# Top 20 Most Frequent Words
freq = Counter(all_words)
print("Top 20 Most Frequent Words:")
print("-" * 35)
for rank, (word, count) in enumerate(freq.most_common(20), 1):
    print(f"  {rank:>2}. {word:<15} -> {count}")

Top 20 Most Frequent Words:
-----------------------------------
   1. good            -> 120
   2. phone           -> 105
   3. camera          -> 81
   4. display         -> 52
   5. best            -> 43
   6. battery         -> 41
   7. performance     -> 32
   8. price           -> 31
   9. quality         -> 30
  10. mobile          -> 26
  11. nice            -> 26
  12. range           -> 22
  13. overall         -> 20
  14. awesome         -> 19
  15. great           -> 18
  16. product         -> 17
  17. also            -> 17
  18. one             -> 17
  19. design          -> 17
  20. motorola        -> 15


---
## Task 3: Feature Engineering
### 3.1 One-Hot Encoding (Document-Level)

In [14]:
from sklearn.preprocessing import OneHotEncoder

all_words_arr = np.array(all_words).reshape(-1, 1)
encoder = OneHotEncoder(sparse_output=False)
encoder.fit(all_words_arr)

doc_vectors = []
for sentence in final_tokens:
    if len(sentence) > 0:
        words = np.array(sentence).reshape(-1, 1)
        encoded = encoder.transform(words)
        doc_vector = encoded.max(axis=0)
    else:
        doc_vector = np.zeros(len(encoder.categories_[0]))
    doc_vectors.append(doc_vector)

doc_vectors = np.array(doc_vectors)
ohe_df = pd.DataFrame(doc_vectors, columns=encoder.categories_[0])
print(f"One-Hot Encoding Matrix Shape: {ohe_df.shape}")
ohe_df.head()

One-Hot Encoding Matrix Shape: (223, 666)


,0,1,10,1010,12,13,18k,19k,1camera,1excellent,...,works,worst,worth,worthc2c,worthy,wow,writ,writing,wrost,zooming
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


### 3.2 Bag of Words (CountVectorizer)

In [15]:
from sklearn.feature_extraction.text import CountVectorizer

sentences = [' '.join(s) for s in final_tokens]

bow = CountVectorizer()
bow_matrix = bow.fit_transform(sentences)
bow_df = pd.DataFrame(bow_matrix.toarray(), columns=bow.get_feature_names_out())
print(f"Bag of Words Matrix Shape: {bow_matrix.shape}")
bow_df.head()

Bag of Words Matrix Shape: (223, 655)


,10,1010,12,13,18k,19k,1camera,1excellent,1st,20000,...,works,worst,worth,worthc2c,worthy,wow,writ,writing,wrost,zooming
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


### 3.3 TF-IDF (TfidfVectorizer)

In [37]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(norm=None, sublinear_tf=True)
tfidf_matrix = tfidf.fit_transform(sentences)
tfidf_df = pd.DataFrame(tfidf_matrix.toarray(), columns=tfidf.get_feature_names_out())
print(f"TF-IDF Matrix Shape: {tfidf_matrix.shape}")
tfidf_df.head()

TF-IDF Matrix Shape: (223, 655)


,10,1010,12,13,18k,19k,1camera,1excellent,1st,20000,...,works,worst,worth,worthc2c,worthy,wow,writ,writing,wrost,zooming
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


---
## Task 4: Comparison Analysis

In [48]:
# Compare key words across OHE, BoW, TF-IDF
important_words = ['good', 'camera', 'phone', 'battery', 'display', 'price']
important_words = [w for w in important_words if w in bow_df.columns]

print("=" * 65)
print("COMPARISON TABLE: OHE vs BoW vs TF-IDF")
print("=" * 65)

comparison_data = {}
for word in important_words:
    comparison_data[word] = {
        'OHE (doc count)': int(ohe_df[word].sum()),
        'BoW (total count)': int(bow_df[word].sum()),
        'TF-IDF (max)': round(tfidf_df[word].max(), 4),
    }

print(pd.DataFrame(comparison_data).T)

print("\n--- Why common words receive lower TF-IDF weight ---")
print("TF-IDF = Term Frequency x Inverse Document Frequency.")
print("Words appearing in many documents get a low IDF value,")
print("which reduces their overall TF-IDF score. This ensures")
print("rare, distinctive words receive higher importance scores.")

COMPARISON TABLE: OHE vs BoW vs TF-IDF
         OHE (doc count)  BoW (total count)  TF-IDF (max)
good                94.0              120.0        3.8987
camera              71.0               81.0        5.0947
phone               87.0              105.0        4.0594
battery             39.0               41.0        4.6100
display             48.0               52.0        4.2664
price               29.0               31.0        5.0971

--- Why common words receive lower TF-IDF weight ---
TF-IDF = Term Frequency x Inverse Document Frequency.
Words appearing in many documents get a low IDF value,
which reduces their overall TF-IDF score. This ensures
rare, distinctive words receive higher importance scores.


In [50]:
# Top 10 and Least 10 words comparison
top10 = freq.most_common(10)
top10_words = [w for w, c in top10 if w in bow_df.columns]

print("TOP 10 FREQUENT WORDS:")
top_comp = pd.DataFrame({
    'Frequency': [freq[w] for w in top10_words],
    'OHE (docs)': [int(ohe_df[w].sum()) for w in top10_words],
    'BoW (count)': [int(bow_df[w].sum()) for w in top10_words],
    'TF-IDF (avg)': [round(tfidf_df[w].mean(), 4) for w in top10_words],
}, index=top10_words)
print(top_comp)

print("\nLEAST 10 FREQUENT WORDS:")
least10 = freq.most_common()[:-11:-1]
least10_words = [w for w, c in least10 if w in bow_df.columns]
least_comp = pd.DataFrame({
    'Frequency': [freq[w] for w in least10_words],
    'OHE (docs)': [int(ohe_df[w].sum()) for w in least10_words],
    'BoW (count)': [int(bow_df[w].sum()) for w in least10_words],
    'TF-IDF (max)': [round(tfidf_df[w].max(), 4) for w in least10_words],
}, index=least10_words)
print(least_comp)

TOP 10 FREQUENT WORDS:
             Frequency  OHE (docs)  BoW (count)  TF-IDF (avg)
good               120          94          120        0.9260
phone              105          87          105        0.8529
camera              81          71           81        0.7367
display             52          48           52        0.5737
best                43          36           43        0.5058
battery             41          39           41        0.4931
performance         32          30           32        0.4191
price               31          29           31        0.4102
quality             30          27           30        0.4016
mobile              26          26           26        0.3633

LEAST 10 FREQUENT WORDS:
            Frequency  OHE (docs)  BoW (count)  TF-IDF (max)
higher              1           1            1        5.7185
slightly            1           1            1        5.7185
scanner             1           1            1        5.7185
finger              1    

---
## Task 5: Sparse Matrix Analysis

In [53]:
print("=" * 55)
print("MATRIX SHAPES")
print("=" * 55)
print(f"One-Hot Encoding : {doc_vectors.shape}")
print(f"Bag of Words     : {bow_matrix.shape}")
print(f"TF-IDF           : {tfidf_matrix.shape}")

print("\n" + "=" * 55)
print("SPARSITY ANALYSIS")
print("=" * 55)

ohe_total = doc_vectors.size
ohe_zeros = (doc_vectors == 0).sum()
ohe_sparsity = (ohe_zeros / ohe_total) * 100

bow_total = bow_matrix.shape[0] * bow_matrix.shape[1]
bow_zeros = bow_total - bow_matrix.nnz
bow_sparsity = (bow_zeros / bow_total) * 100

tfidf_total = tfidf_matrix.shape[0] * tfidf_matrix.shape[1]
tfidf_zeros = tfidf_total - tfidf_matrix.nnz
tfidf_sparsity = (tfidf_zeros / tfidf_total) * 100

sparsity = pd.DataFrame({
    'Matrix': ['One-Hot', 'Bag of Words', 'TF-IDF'],
    'Shape': [str(doc_vectors.shape), str(bow_matrix.shape), str(tfidf_matrix.shape)],
    'Total Elements': [ohe_total, bow_total, tfidf_total],
    'Zeros': [int(ohe_zeros), int(bow_zeros), int(tfidf_zeros)],
    'Sparsity %': [round(ohe_sparsity,2), round(bow_sparsity,2), round(tfidf_sparsity,2)]
})
print(sparsity.to_string(index=False))

MATRIX SHAPES
One-Hot Encoding : (223, 666)
Bag of Words     : (223, 655)
TF-IDF           : (223, 655)

SPARSITY ANALYSIS
      Matrix      Shape  Total Elements  Zeros  Sparsity %
     One-Hot (223, 666)          148518 146647       98.74
Bag of Words (223, 655)          146065 144211       98.73
      TF-IDF (223, 655)          146065 144211       98.73


### Why Sparse Matrices Are Inefficient for Large-Scale Systems

1. **Memory Waste** — Millions of stored zeros consume RAM without adding any information.
2. **Slow Computation** — Dense matrix operations waste CPU cycles multiplying by zero.
3. **Curse of Dimensionality** — Each unique word adds a feature dimension, degrading model performance as vocabulary grows.
4. **No Semantic Information** — Each word is an independent axis; synonyms/context are ignored.

**Solution:** Use CSR/CSC sparse formats (sklearn default), or move to dense embeddings (Word2Vec, BERT).

---
## Task 6: Real-World Questions

### Q1: Why Bag of Words fails in understanding semantic meaning

| Problem | Example |
|---|---|
| **Word order lost** | "dog bites man" vs "man bites dog" → identical BoW |
| **Negation invisible** | "good" vs "not good" → same after stopword removal |
| **Synonyms treated differently** | "good" and "excellent" have zero similarity |
| **Polysemy ignored** | "bank" (money) vs "bank" (river) → same feature |
| **Phrases broken** | "hot dog" (food) vs "hot weather" → words split apart |
| **Equal weighting** | "battery" and "the" both count as 1 |

### Q2: When to use BoW and TF-IDF in industry

**BoW is best for:**
- Spam filtering (keyword presence matters)
- Quick baseline text classifiers
- Topic labelling when frequency drives meaning

**TF-IDF is best for:**
- Search engines and document ranking
- News/article classification
- Information retrieval where word importance matters
- Any task where common words should be down-weighted

### Q3: Limitations of TF-IDF in real applications

1. No understanding of word order or grammar
2. Synonyms ("good" / "excellent") treated as completely unrelated
3. High dimensionality with large vocabularies
4. Cannot handle out-of-vocabulary (OOV) words
5. Same word with different meanings (polysemy) gets one score
6. Static weights — cannot learn from context

**Modern alternatives:** Word2Vec, GloVe, FastText, BERT, GPT embeddings.

---
## Task 7: Mini Use Case — Sentiment Classification
Positive vs Negative review classification using Naive Bayes and Logistic Regression.
Compare BoW vs TF-IDF features.

In [54]:
from textblob import TextBlob

labels = []
for sentence in sentences:
    polarity = TextBlob(sentence).sentiment.polarity
    labels.append(1 if polarity >= 0 else 0)

labels = np.array(labels)
print(f"Total samples  : {len(labels)}")
print(f"Positive (1)   : {(labels == 1).sum()}")
print(f"Negative (0)   : {(labels == 0).sum()}")

Total samples  : 223
Positive (1)   : 218
Negative (0)   : 5


In [55]:
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score

X_train_bow, X_test_bow, y_train, y_test = train_test_split(
    bow_matrix, labels, test_size=0.2, random_state=42
)
X_train_tfidf, X_test_tfidf, _, _ = train_test_split(
    tfidf_matrix, labels, test_size=0.2, random_state=42
)
print(f"Train: {X_train_bow.shape[0]}  |  Test: {X_test_bow.shape[0]}")

Train: 178  |  Test: 45


In [56]:
# --- Naive Bayes + BoW ---
nb_bow = MultinomialNB()
nb_bow.fit(X_train_bow, y_train)
pred_nb_bow = nb_bow.predict(X_test_bow)

print("=" * 55)
print("NAIVE BAYES + BoW")
print("=" * 55)
print(f"Accuracy: {accuracy_score(y_test, pred_nb_bow):.4f}")
print(classification_report(y_test, pred_nb_bow,
      target_names=['Negative','Positive'], zero_division=0))

NAIVE BAYES + BoW
Accuracy: 0.9333
              precision    recall  f1-score   support

    Negative       0.00      0.00      0.00         0
    Positive       1.00      0.93      0.97        45

    accuracy                           0.93        45
   macro avg       0.50      0.47      0.48        45
weighted avg       1.00      0.93      0.97        45



In [57]:
# --- Naive Bayes + TF-IDF ---
nb_tfidf = MultinomialNB()
nb_tfidf.fit(X_train_tfidf, y_train)
pred_nb_tfidf = nb_tfidf.predict(X_test_tfidf)

print("=" * 55)
print("NAIVE BAYES + TF-IDF")
print("=" * 55)
print(f"Accuracy: {accuracy_score(y_test, pred_nb_tfidf):.4f}")
print(classification_report(y_test, pred_nb_tfidf,
      target_names=['Negative','Positive'], zero_division=0))

NAIVE BAYES + TF-IDF
Accuracy: 0.6667
              precision    recall  f1-score   support

    Negative       0.00      0.00      0.00         0
    Positive       1.00      0.67      0.80        45

    accuracy                           0.67        45
   macro avg       0.50      0.33      0.40        45
weighted avg       1.00      0.67      0.80        45



In [58]:
# --- Logistic Regression + BoW ---
lr_bow = LogisticRegression(max_iter=1000)
lr_bow.fit(X_train_bow, y_train)
pred_lr_bow = lr_bow.predict(X_test_bow)

print("=" * 55)
print("LOGISTIC REGRESSION + BoW")
print("=" * 55)
print(f"Accuracy: {accuracy_score(y_test, pred_lr_bow):.4f}")
print(classification_report(y_test, pred_lr_bow,
      labels=[0, 1],
      target_names=['Negative','Positive'], zero_division=0))

LOGISTIC REGRESSION + BoW
Accuracy: 1.0000
              precision    recall  f1-score   support

    Negative       0.00      0.00      0.00         0
    Positive       1.00      1.00      1.00        45

    accuracy                           1.00        45
   macro avg       0.50      0.50      0.50        45
weighted avg       1.00      1.00      1.00        45



In [59]:
# --- Logistic Regression + TF-IDF ---
lr_tfidf = LogisticRegression(max_iter=1000)
lr_tfidf.fit(X_train_tfidf, y_train)
pred_lr_tfidf = lr_tfidf.predict(X_test_tfidf)

print("=" * 55)
print("LOGISTIC REGRESSION + TF-IDF")
print("=" * 55)
print(f"Accuracy: {accuracy_score(y_test, pred_lr_tfidf):.4f}")
print(classification_report(y_test, pred_lr_tfidf,
      labels=[0, 1],
      target_names=['Negative','Positive'], zero_division=0))

LOGISTIC REGRESSION + TF-IDF
Accuracy: 1.0000
              precision    recall  f1-score   support

    Negative       0.00      0.00      0.00         0
    Positive       1.00      1.00      1.00        45

    accuracy                           1.00        45
   macro avg       0.50      0.50      0.50        45
weighted avg       1.00      1.00      1.00        45



In [60]:
# ============================================================
# FINAL COMPARISON TABLE
# ============================================================
from sklearn.metrics import precision_score, recall_score

results = pd.DataFrame({
    'Model': ['Naive Bayes','Naive Bayes','Logistic Reg','Logistic Reg'],
    'Features': ['BoW','TF-IDF','BoW','TF-IDF'],
    'Accuracy': [
        round(accuracy_score(y_test, pred_nb_bow), 4),
        round(accuracy_score(y_test, pred_nb_tfidf), 4),
        round(accuracy_score(y_test, pred_lr_bow), 4),
        round(accuracy_score(y_test, pred_lr_tfidf), 4),
    ],
    'F1-Score': [
        round(f1_score(y_test, pred_nb_bow, zero_division=0), 4),
        round(f1_score(y_test, pred_nb_tfidf, zero_division=0), 4),
        round(f1_score(y_test, pred_lr_bow, zero_division=0), 4),
        round(f1_score(y_test, pred_lr_tfidf, zero_division=0), 4),
    ]
})

print("=" * 60)
print("FINAL MODEL COMPARISON: BoW vs TF-IDF")
print("=" * 60)
print(results.to_string(index=False))
best = results.loc[results['Accuracy'].idxmax()]
print(f"\nBest: {best['Model']} + {best['Features']} ({best['Accuracy']:.2%})")

FINAL MODEL COMPARISON: BoW vs TF-IDF
       Model Features  Accuracy  F1-Score
 Naive Bayes      BoW    0.9333    0.9655
 Naive Bayes   TF-IDF    0.6667    0.8000
Logistic Reg      BoW    1.0000    1.0000
Logistic Reg   TF-IDF    1.0000    1.0000

Best: Logistic Reg + BoW (100.00%)


---
## Conclusions

1. **Preprocessing** is critical — cleaning, stopword removal, and normalisation improve feature quality significantly.
2. **One-Hot Encoding** creates binary presence vectors; useful for understanding word presence but ignores frequency.
3. **Bag of Words** captures frequency but treats all words equally and loses word order.
4. **TF-IDF** improves on BoW by down-weighting common words and highlighting rare, distinctive terms.
5. **Sparsity** exceeds 95% across all representations, making compressed formats essential.
6. **Sentiment Classification** — even simple classifiers achieve strong accuracy with basic text features.
7. **Limitations** — All three methods ignore word order, context, and semantics. Modern NLP uses dense embeddings (Word2Vec, BERT) to overcome these gaps.